In [94]:
import pandas as pd
import pyreadstat
# set pandas display options to show all columns
pd.set_option('display.max_columns', None)

# import basics
import pandas as pd
import numpy as np

# import tools
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.base import BaseEstimator, TransformerMixin

# import models
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from lightgbm import LGBMRegressor

# import viz
import altair as alt
alt.renderers.enable('mimetype') # for altair plots to be properly rendered on GH


RendererRegistry.enable('mimetype')

In [85]:
df_school, meta_school = pyreadstat.read_sav('../data/Data_PIRLS16(sav)/P4_SCHOOL16.sav')
df_student, meta_student = pyreadstat.read_sav('../data/Data_PIRLS16(sav)/P4_STUDENT16.sav')
df_teacher, meta_teacher = pyreadstat.read_sav('../data/Data_PIRLS16(sav)/P4_TEACHER16.sav')
df_link, meta_link = pyreadstat.read_sav('../data/Data_PIRLS16(sav)/P4_STD_TCH_LINK16.sav')


In [86]:
# cols to generate y
y_gen_cols = ['ASRREA01', 'ASRREA02', 'ASRREA03', 'ASRREA04', 'ASRREA05']
# cols to drop because they might lead to leakage
cols_drop = y_gen_cols + ['ASRLIT01', 'ASRLIT02', 'ASRLIT03', 'ASRLIT04', 'ASRLIT05', 'ASRINF01', 'ASRINF02', 'ASRINF03', 'ASRINF04', 'ASRINF05', 'ASRIIE01', 'ASRIIE02', 'ASRIIE03', 'ASRIIE04', 'ASRIIE05', 'ASRRSI01', 'ASRRSI02', 'ASRRSI03', 'ASRRSI04', 'ASRRSI05', ]

In [ ]:
def create_y(df, y_gen_cols):
    """Create y column by averaging the columns in y_gen_cols"""
    df['y'] = df[y_gen_cols].mean(axis=1)
    return df

def remove_cols(df, cols_drop):
    """Remove columns from the dataframe if they exist"""
    # drop column if it exists
    to_drop = [col for col in cols_drop if col in df.columns]
    # drop columns
    df = df.drop(columns=to_drop)
    return df

def merge_dfs(df_student, df_new, on):
    """ Merge df_new into df_student on the given column"""
    # record original number of rows
    original_rows = df_student.shape[0]
    # drop overlapping columns in df_new except for the merge column
    cols_to_drop = [col for col in df_new.columns if col in df_student.columns and col != on]
    df_new = df_new.drop(columns=cols_to_drop)
    # merge the dataframes
    df_student = df_student.merge(df_new, on=on, how='left', suffixes=('', '_new'))
    # drop the new columns that are now duplicates
    cols_to_drop = [col for col in df_student.columns if col.endswith('_new')]
    df_student = df_student.drop(columns=cols_to_drop)
    # assert that the number of rows is the same
    assert df_student.shape[0] == original_rows, f"Number of rows changed from {original_rows} to {df_student.shape[0]}"
    return df_student


In [87]:
df_student.shape

(4425, 152)

In [88]:
def create_X_and_y(df_student, df_teacher, df_school, df_link):
    """ Create a dataframe with the relevant columns from the student, teacher, school, and link dataframes"""
    global y_gen_cols, cols_drop
    df_student = create_y(df_student, y_gen_cols)
    df_student = remove_cols(df_student, cols_drop)
    df_school = remove_cols(df_school, cols_drop)
    df_teacher = remove_cols(df_teacher, cols_drop)
    df_link = remove_cols(df_link, cols_drop)
    
    df = merge_dfs(df_student, df_school, on='IDSCHOOL')
    df = merge_dfs(df, df_link, on='IDSTUD')
    df = merge_dfs(df, df_teacher, on='IDTEALIN')
    
    y = df['y']
    X = df.drop(columns=['y'])
    return X, y

In [89]:
X, y = create_X_and_y(df_student, df_teacher, df_school, df_link)
X.shape, y.shape

((4425, 390), (4425,))

In [91]:
X_train, X_test, y_train, y_test = train_test_split(X,
                                                    y,
                                                    test_size=0.3,
                                                    random_state=9527)


In [ ]:

# create the list of cols to drop based on missing rate
missing_rates = (X_train.isnull().sum()/X_train.shape[0]).sort_values(ascending=False)[:20]
miss_rate_bar = 0.8

missing_cols = missing_rates[missing_rates > miss_rate_bar].index.tolist()



# # a custom sklearn pipeline function step to remove columns that are missing too much data
# class ColumnDropper(BaseEstimator, TransformerMixin):
#     def __init__(self, miss_rate_bar=0.8):
#         self.miss_rate_bar = miss_rate_bar
#         self.is_fitted = False
#         pass
    
#     def fit(self, X, y=None):
#         missing_rates = X.isnull().sum()/X.shape[0]
#         self.missing_cols = missing_rates[missing_rates > self.miss_rate_bar].index.tolist()
#         self.is_fitted = True
#         return self

#     def transform(self, X):
#         assert self.is_fitted, "The ColumnDropper has not been fitted yet. Please call fit() before transform()."
#         return X.drop(columns=self.cols_to_drop)